In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import os
import snowflake.snowpark as snowpark
from snowflake.snowpark.session import Session
from snowflake.snowpark.functions import col
from dotenv import load_dotenv

load_dotenv()

In [ ]:
connection_parameters = {
    "account": os.getenv(SNOWFLAKE_ACCOUNT),
    "user": os.getenv(SNOWFLAKE_USER),
    "authenticator": os.getenv(SNOWFLAKE_AUTHENTICATOR),
    "insecure_mode":True,
}

session = Session.builder.configs(connection_parameters).create()

In [ ]:
# Sales Channel GAs

query = f"""
SELECT
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD,
    SUM(GROSS_ACTIVATION_QUANTITY) AS TOTAL_GROSS_ACTIVATION
FROM
    {os.getenv("SALES_TABLE")}
WHERE
    ACTIVATION_DATE BETWEEN DATE_TRUNC('QUARTER', CURRENT_DATE) AND CURRENT_DATE
GROUP BY
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD
ORDER BY
    ACTIVATION_DATE,
    SLSOPS_CHANNEL_RECORD
"""
ga_snowflake = session.sql(query).to_pandas()


In [ ]:

slsops_to_sales_channel = {
    'Direct Distribution Partner': 'Indirect',
    'Axiom Indirect': 'Digital',
    'Dish Internal': 'Digital',
    'Web Sales': 'Digital',
    'Amazon': 'Amazon',
    'Best Buy': 'National Retail',
    'National Retailer': 'National Retail',
    'WalMart': 'National Retail',
    'Target': 'National Retail',
    'The Kroger Co.': 'National Retail',
    'Cross Sell': 'Cross Sell',
    'Direct': 'Telesales'
}
ga['Channel'] = ga_snowflake['SLSOPS_CHANNEL_RECORD'].map(slsops_to_sales_channel)